In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from skimage import io
from skimage.filters import threshold_otsu
import panel as pn
import plotly.graph_objects as go
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, PolySelectTool, LassoSelectTool, Span

pn.extension('plotly', sizing_mode="stretch_width")

class FluoJo:
    def __init__(self):
        self.current_dir = os.getcwd()
        self.data_dir = Path("./Cellpose_output")
        self.output_dir = Path("./interactive_quantification_results")
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.csv_suffix = "corrected.csv"
        self.mask_suffix = "proce3Dflow__cellposed_filtered_cp_masks.tif"
        
        self.df = None
        self.gating_history = {} 
        self.current_global_selection = [] 
        
        # --- UI Elements ---
        self.file_selector = pn.widgets.Select(name='Select File Pair', options=self.get_file_pairs())
        self.base_population = pn.widgets.Select(name='Base Population (Parent Gate)', options=['All_Cells'])
        
        self.highlight_subgroups = pn.widgets.MultiChoice(
            name='Highlight Subgroups (2D only)', options=[], 
            placeholder='Select gates to overlay...'
        )
        
        # Calculate Co-positivity button
        self.calc_copos_btn = pn.widgets.Button(name='Calculate Co-positivity (Base ∩ Highlights)', button_type='success')
        self.calc_copos_btn.on_click(self.calculate_copositive)
        
        self.feature_x = pn.widgets.Select(name='Feature X')
        self.feature_y = pn.widgets.Select(name='Feature Y (Select "None" for 1D)', options=['None'])
        
        self.add_plot_btn = pn.widgets.Button(name='Add Plot to Workspace', button_type='warning')
        self.add_plot_btn.on_click(self.add_2d_plot)
        
        self.clear_plots_btn = pn.widgets.Button(name='Clear Workspace', button_type='danger')
        self.clear_plots_btn.on_click(self.clear_workspace)
        
        self.gate_name_input = pn.widgets.TextInput(name='Gate Name', placeholder='e.g., Epiblast_High_GFP')
        
        self.save_gate_btn = pn.widgets.Button(name='Save Current Gate', button_type='primary')
        self.save_gate_btn.on_click(self.save_current_gate)
        
        # Remove Gate UI
        self.remove_gate_select = pn.widgets.Select(name='Select Gate to Remove', options=[])
        self.remove_gate_btn = pn.widgets.Button(name='Remove Saved Gate', button_type='danger')
        self.remove_gate_btn.on_click(self.remove_gate)
        
        self.export_csv_btn = pn.widgets.Button(name='Export Analysis CSV', button_type='success')
        self.export_csv_btn.on_click(self.export_data)
        
        self.status_text = pn.pane.Markdown("Status: Ready to load data.")
        
        self.file_selector.param.watch(self.load_data, 'value')
        
        self.plots_workspace = pn.FlexBox(align_items='start', justify_content='start')
        self.plot_3d_pane = pn.pane.Plotly()
        self.stats_table = pn.widgets.DataFrame(height=150, sizing_mode="stretch_width")

    def get_file_pairs(self):
        if not self.data_dir.exists():
            return {"(Data Directory Not Found)": None}
        csv_files = list(self.data_dir.glob(f"*{self.csv_suffix}"))
        pairs = {"-- Select a File --": None}
        for csv in csv_files:
            prefix = csv.name.replace(self.csv_suffix, "")
            expected_tif = self.data_dir / f"{prefix}{self.mask_suffix}"
            if expected_tif.exists():
                pairs[prefix] = {"csv": csv, "tif": expected_tif}
        return pairs

    def load_data(self, event):
        selection = event.new
        if not selection: return
        try:
            self.status_text.object = "Status: Loading and validating data..."
            self.plots_workspace[:] = [] 
            self.current_global_selection = []
            
            csv_path = selection["csv"]
            tif_path = selection["tif"]
            
            self.df = pd.read_csv(csv_path).copy()
            self.df['_orig_index'] = np.arange(len(self.df))
            
            mask = io.imread(tif_path)
            num_tif_cells = np.max(mask)
            
            if len(self.df) != num_tif_cells:
                self.status_text.object = f"⚠️ **Mismatch!** CSV: {len(self.df)}, TIF: {num_tif_cells} cells."
            else:
                self.status_text.object = f"✅ Success! Loaded {len(self.df)} cells."
            
            self.gating_history = {'All_Cells': np.ones(len(self.df), dtype=bool)}
            self.update_gate_selectors()
            self.base_population.value = 'All_Cells'
            
            numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
            if '_orig_index' in numeric_cols: numeric_cols.remove('_orig_index')
            
            self.feature_x.options = numeric_cols
            self.feature_y.options = ['None'] + numeric_cols
            
            if len(numeric_cols) > 0: self.feature_x.value = numeric_cols[0]
                
            self.update_stats()
            self.update_3d_plot() 
        except Exception as e:
            self.status_text.object = f"❌ **Error:** {str(e)}"

    def update_gate_selectors(self):
        opts = list(self.gating_history.keys())
        self.base_population.options = opts
        self.highlight_subgroups.options = [o for o in opts if o != 'All_Cells']
        self.remove_gate_select.options = [o for o in opts if o != 'All_Cells']

    def clear_workspace(self, event):
        self.plots_workspace[:] = []
        self.current_global_selection = []
        self.update_3d_plot()
        self.status_text.object = "🧹 **Workspace cleared.**"

    def remove_gate(self, event):
        gate_name = self.remove_gate_select.value
        if gate_name and gate_name in self.gating_history:
            del self.gating_history[gate_name]
            self.update_gate_selectors()
            self.update_stats()
            self.status_text.object = f"🗑️ **Removed gate: {gate_name}**"

    def calculate_copositive(self, event):
        base_gate = self.base_population.value
        subgroups = self.highlight_subgroups.value
        
        if not subgroups:
            self.status_text.object = "⚠️ **Please select at least one highlighted subgroup to calculate co-positivity.**"
            return
            
        # Start with the base mask
        combined_mask = self.gating_history[base_gate].copy()
        
        # Intersect with all selected subgroups
        for sg in subgroups:
            combined_mask = combined_mask & self.gating_history[sg]
            
        # Create the requested name format
        new_gate_name = f"{base_gate} -> {' + '.join(subgroups)}"
        
        # Save to history
        self.gating_history[new_gate_name] = combined_mask
        self.update_gate_selectors()
        self.update_stats()
        
        cell_count = combined_mask.sum()
        self.status_text.object = f"✅ **Calculated co-positivity: {new_gate_name} ({cell_count} cells)**"

    def add_2d_plot(self, event=None):
        if self.df is None or not self.feature_x.value: return
            
        x_col = self.feature_x.value
        y_col = self.feature_y.value
        base_gate = self.base_population.value
        subgroups_to_highlight = self.highlight_subgroups.value
        
        try:
            base_mask = self.gating_history[base_gate]
            subset_df = self.df[base_mask]
            source = ColumnDataSource(subset_df.to_dict('list'))
            
            def selection_callback(attr, old, new):
                if not new:
                    self.current_global_selection = []
                    self.update_3d_plot(base_mask=base_mask, parent_name=base_gate)
                    return
                self.current_global_selection = [source.data['_orig_index'][i] for i in new]
                self.update_3d_plot(base_mask=base_mask, parent_name=base_gate, highlight_global_indices=self.current_global_selection)

            source.selected.on_change('indices', selection_callback)
            threshold_controls = pn.Row()
            
            if y_col == 'None':
                p = figure(title=f"1D: {x_col} ({base_gate})", tools="pan,wheel_zoom,reset", width=350, height=300)
                p.xaxis.axis_label = x_col
                p.yaxis.axis_label = "Frequency"
                
                data = subset_df[x_col].dropna().values
                if len(data) > 0:
                    hist, edges = np.histogram(data, bins=50)
                    p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], fill_color="skyblue", line_color="white")
                    bin_centers = (edges[:-1] + edges[1:]) / 2
                    try:
                        otsu_val = threshold_otsu(hist=(hist, bin_centers))
                        p.line([otsu_val, otsu_val], [0, max(hist)], color='orange', line_width=2, legend_label=f"Otsu: {otsu_val:.2f}")
                        default_val = float(otsu_val)
                    except: default_val = 0.0
                
                vline = Span(location=default_val, dimension='height', line_color='red', line_dash='dashed', line_width=2)
                p.add_layout(vline)
                tx = pn.widgets.FloatInput(name=f'{x_col} Thresh', value=default_val, width=120)
                tx.param.watch(lambda e: setattr(vline, 'location', e.new), 'value')
                threshold_controls.append(tx)
                
            else:
                p = figure(title=f"2D: {x_col} vs {y_col} ({base_gate})", tools="pan,wheel_zoom,reset", width=350, height=300)
                p.xaxis.axis_label = x_col
                p.yaxis.axis_label = y_col
                
                # Plot Base population first
                p.scatter(x=x_col, y=y_col, source=source, size=4, alpha=0.3, color='gray', nonselection_alpha=0.1, selection_color="red")
                
                colors = ['green', 'yellow', 'magenta', 'cyan', 'orange']
                for i, subgroup in enumerate(subgroups_to_highlight):
                    sub_mask = self.gating_history[subgroup]
                    final_mask = base_mask & sub_mask
                    sub_df = self.df[final_mask]
                    if not sub_df.empty:
                        sub_source = ColumnDataSource(sub_df.to_dict('list'))
                        color = colors[i % len(colors)]
                        p.scatter(x=x_col, y=y_col, source=sub_source, size=5, 
                                  alpha=0.5, color=color, legend_label=subgroup)
                
                if p.legend:
                    p.legend.click_policy="hide"
                    p.legend.label_text_font_size = "8pt"

                p.add_tools(PolySelectTool(), LassoSelectTool())
                
                def_x, def_y = float(subset_df[x_col].mean()) if not subset_df.empty else 0.0, float(subset_df[y_col].mean()) if not subset_df.empty else 0.0
                vline = Span(location=def_x, dimension='height', line_color='red', line_dash='dashed', line_width=2)
                hline = Span(location=def_y, dimension='width', line_color='green', line_dash='dashed', line_width=2)
                p.add_layout(vline); p.add_layout(hline)
                
                tx = pn.widgets.FloatInput(name='X Thresh', value=def_x, width=110)
                ty = pn.widgets.FloatInput(name='Y Thresh', value=def_y, width=110)
                tx.param.watch(lambda e: setattr(vline, 'location', e.new), 'value')
                ty.param.watch(lambda e: setattr(hline, 'location', e.new), 'value')
                threshold_controls.extend([tx, ty])
            
            remove_btn = pn.widgets.Button(name='✖ Remove Plot', button_type='danger', width=100)
            plot_card = pn.Column(pn.Row(threshold_controls, pn.Spacer(), remove_btn), pn.pane.Bokeh(p), styles={'background': '#f5f5f5', 'padding': '10px'})
            remove_btn.on_click(lambda e: self.plots_workspace.remove(plot_card))
            self.plots_workspace.append(plot_card)
        except Exception as e: self.status_text.object = f"❌ **Error:** {str(e)}"

    def update_3d_plot(self, base_mask=None, parent_name="All_Cells", highlight_global_indices=None):
        if self.df is None: return
        if base_mask is None: base_mask = np.ones(len(self.df), dtype=bool)
        
        subset_df = self.df[base_mask]
        traces = []
        
        # Background
        if not np.all(base_mask):
            other = self.df[~base_mask]
            traces.append(go.Scatter3d(x=other['Centroid_X'], y=other['Centroid_Y'], z=other['Centroid_Z'], mode='markers', marker=dict(size=2, color='gray', opacity=0.2), name="Context"))

        # Parent
        traces.append(go.Scatter3d(x=subset_df['Centroid_X'], y=subset_df['Centroid_Y'], z=subset_df['Centroid_Z'], mode='markers', marker=dict(size=3, color='steelblue', opacity=0.5), name=parent_name))
        
        # Current Selection
        if highlight_global_indices:
            hl = self.df.iloc[highlight_global_indices]
            traces.append(go.Scatter3d(x=hl['Centroid_X'], y=hl['Centroid_Y'], z=hl['Centroid_Z'], mode='markers', marker=dict(size=4, color='red', opacity=1.0), name="Selection"))

        fig = go.Figure(data=traces)
        fig.update_layout(
            title=f"3D Map (Parent: {parent_name})",
            scene=dict(
                aspectmode='cube',
                aspectratio=dict(x=1, y=1, z=1)
            ),
            width=500, height=450, margin=dict(l=0, r=0, b=0, t=30), uirevision='constant'
        )
        self.plot_3d_pane.object = fig

    def save_current_gate(self, event):
        name = self.gate_name_input.value.strip()
        if not name or not self.current_global_selection: return
        mask = np.zeros(len(self.df), dtype=bool)
        mask[self.current_global_selection] = True
        self.gating_history[name] = mask
        self.gate_name_input.value = ""
        self.update_gate_selectors()
        self.update_stats()
        self.status_text.object = f"✅ **Saved: {name}**"

    def update_stats(self):
        stats = [{"Gate": k, "Cell_Count": v.sum()} for k, v in self.gating_history.items()]
        self.stats_table.value = pd.DataFrame(stats)

    def export_data(self, event):
        if self.df is None or not self.file_selector.value: return
        
        out_df = self.df.drop(columns=['_orig_index'], errors='ignore').copy()
        for gate, mask in self.gating_history.items():
            if gate != 'All_Cells': out_df[gate] = mask.astype(int)
            
        # Extract the prefix of the currently selected file
        prefix = self.file_selector.value['csv'].name.replace(self.csv_suffix, "")
        
        # Clean up any trailing underscore if present to avoid names like 'Sample1__analysis_results.csv'
        if prefix.endswith('_'):
            prefix = prefix[:-1]
            
        out_filename = f"{prefix}_analysis_results.csv"
        out_path = self.output_dir / out_filename
        
        out_df.to_csv(out_path, index=False)
        self.status_text.object = f"💾 **Saved to: {out_path.name}**"

        # --- NEW: Export Layout as HTML ---
        if len(self.plots_workspace) > 0 or self.plot_3d_pane.object is not None:
            try:
                html_filename = f"{prefix}_workspace_plots.html"
                html_path = self.output_dir / html_filename
                
                # Create a temporary layout containing all current plots and 3D map
                export_layout = pn.Column(
                    f"## Spatial Gating Analysis Plots: {prefix}",
                    pn.Row(self.plots_workspace, self.plot_3d_pane)
                )
                
                # Save as interactive HTML
                export_layout.save(str(html_path), title=f"{prefix} Gating Plots")
                status_msg = f"💾 **Saved CSV:** {out_filename} <br> **Saved Plots:** {html_filename}"
            except Exception as e:
                status_msg = f"💾 **Saved CSV:** {out_filename} <br> ⚠️ Plot export failed: {str(e)}"
                
        self.status_text.object = status_msg

    def view(self):
        controls = pn.Column(
            "### Spatial Gating Tool", self.file_selector, self.status_text, pn.layout.Divider(),
            self.base_population, self.highlight_subgroups,
            self.calc_copos_btn, 
            pn.Row(self.feature_x, self.feature_y), pn.Row(self.add_plot_btn, self.clear_plots_btn),
            pn.layout.Divider(), self.gate_name_input, self.save_gate_btn,
            pn.layout.Divider(), "**Manage Gates**", self.remove_gate_select, self.remove_gate_btn,
            self.stats_table, self.export_csv_btn, width=350
        )
        return pn.Row(controls, self.plots_workspace, self.plot_3d_pane)

tool = FluoJo()
tool.view().show()

Launching server at http://localhost:56241
